In [81]:
import torch as t
from torch import nn
import torch.nn.functional as F
import opt_einsum as oe
from typing import List, Tuple, Any
from dataclasses import dataclass

device = "xpu"
t.random.manual_seed(42)

In [82]:
def tt_svd(X: t.Tensor):

    def k_unfolding(X: t.Tensor, k=2):
        sh = X.shape
        return X.reshape(sh[:k].numel(), sh[k:].numel())

    def flip_k_leg(M: t.Tensor, shapes, k):
        return M.reshape(M.shape[0] * shapes[k-1], shapes[k:].numel())

    N = X.ndim
    shapes = X.shape
    cores = []
    U, S, VT = t.linalg.svd(k_unfolding(X, k=1), full_matrices=False)
    cores.append(U.unsqueeze(0))
    for k in range(2, N):
        r = VT.shape[0]
        U, S, VT = t.linalg.svd(flip_k_leg(
            S[:, None] * VT, shapes, k), full_matrices=False)
        cores.append(U.reshape(r, U.shape[0] // r, U.shape[1]))
    cores.append((S[:, None] * VT).unsqueeze(-1))
    return cores

In [83]:
def tt_2_tensor(cores):
    """
    Contract a chain of 3-rd order cores
    """
    # assert cores[0].shape[0] == 1 and cores[-1].shape[-1] == 1, \
    #     "it is not a tt"
    N = len(cores)
    offset = N + 1
    expr = []
    res = [0]
    for k, G in enumerate(cores):
        expr += [G, (k, k + offset, k + 1)]
        res += [k + offset]
    res += [N]
    return t.einsum(*expr, res)

In [84]:
def tt_2_matrix(cores: List[t.Tensor]):
    """
    Contract a chain of 3-rd order cores, then reshape to a matrix
    """
    assert len(cores) % 2 == 0, "# cores is odd"
    assert cores[0].shape[0] == 1 and cores[-1].shape[-1] == 1, \
        "it is not a tt"
    d = len(cores) // 2
    X = tt_2_tensor(cores)
    sh = X.shape
    return X.reshape(sh[:d+1].numel(), sh[d+1:].numel())

In [85]:
def build_cores_gaus(shape: t.Size, rank: t.Size):
    """
    Initialize TT cores with given shape and ranks from standard normal
    """
    N = len(shape)
    cores = []
    for n in range(1, N + 1):
        cores.append(t.randn(rank[n-1], shape[n-1], rank[n]))
    return cores

In [86]:
@dataclass
class TTLinearConfig:
    in_shape: t.Size  # I_1, ..., I_d
    out_shape: t.Size  # J_1, ..., J_d
    rank: t.Size  # 1, R_1, ..., R_2d-1, 1
    threshold: float = 1e-2
    adaptive: bool = True
    bias: bool = True
    dtype: t.dtype = t.float32

    def __post_init__(self):
        self.N = len(self.in_shape) + len(self.out_shape)
        assert len(self.rank) == self.N + \
            1, f"TT-rank: expected={self.N + 1}, given={len(self.rank)}"
        assert self.rank[0] == self.rank[-1] == 1, f"It is not TT format: R_0 != 1 or R_-1 != 1"

In [87]:
def A_i(i: int, *cores: List[t.Tensor]):
    return tt_2_tensor(cores[:i])


def B_i(i: int, *cores: List[t.Tensor]):
    d = len(cores) // 2
    return tt_2_tensor(cores[d:d+i])


def A_inv_i(i: int, *cores: List[t.Tensor]):
    d = len(cores) // 2
    return tt_2_tensor(cores[d-i:d])


def B_inv_i(i: int, *cores: List[t.Tensor]):
    d = len(cores) // 2
    return tt_2_tensor(cores[2*d-i:])

In [88]:
class TTMatVec(t.autograd.Function):

    @staticmethod
    def A_i(i: int, *cores: List[t.Tensor]):
        return tt_2_tensor(cores[:i])

    @staticmethod
    def B_i(i: int, *cores: List[t.Tensor]):
        d = len(cores) // 2
        return tt_2_tensor(cores[d:d+i])

    @staticmethod
    def A_inv_i(i: int, *cores: List[t.Tensor]):
        d = len(cores) // 2
        return tt_2_tensor(cores[d-i:d])

    @staticmethod
    def B_inv_i(i: int, *cores: List[t.Tensor]):
        d = len(cores) // 2
        return tt_2_tensor(cores[2*d-i:])

    @staticmethod
    def forward(ctx, X: t.Tensor, *cores: List[t.Tensor]):
        d = len(cores) // 2
        A_d = TTMatVec.A_i(d, *cores)
        B_d = TTMatVec.B_i(d, *cores)
        a_sh = A_d.shape
        b_sh = B_d.shape
        T_1 = X @ A_d.reshape((a_sh[:-1].numel(), a_sh[-1]))
        Y = T_1 @ B_d.reshape((b_sh[0], b_sh[1:].numel()))
        ctx.save_for_backward(X, T_1, *cores)
        return Y

    @staticmethod
    def backward(ctx, *g_Y):
        X, T_1, *cores = ctx.saved_tensors
        d = len(cores) // 2
        r_d = t.Size([cores[d-1].shape[2]])
        in_shape, out_shape = t.Size([]), t.Size([])
        for n in range(d):
            in_shape += t.Size([cores[n].shape[1]])
        for n in range(d, 2*d):
            out_shape += t.Size([cores[n].shape[1]])

        # =============

        # g_X
        A_d = TTMatVec.A_i(d, *cores)
        B_d = TTMatVec.B_i(d, *cores)
        a_sh = A_d.shape
        b_sh = B_d.shape

        U_1 = g_Y[0] @ B_d.reshape((b_sh[0], b_sh[1:].numel())).T
        g_X = U_1 @ A_d.reshape((a_sh[:-1].numel(), a_sh[-1])).T

        # =============

        # g_G_i: i <= d
        U_2 = U_1.T @ X
        g_G_left = []

        # i = 1
        i = 1
        expr_U_2 = [0] + [k for k in range(1, d + 1)]
        expr_A_inv_i = [2 * d + 4] + [k for k in range(i + 1, d + 1)] + [0]
        expr_g_G = [i, 2 * d + 4]

        g_G_1 = t.einsum(
            U_2.reshape(r_d + in_shape), expr_U_2,
            TTMatVec.A_inv_i(d-i, *cores), expr_A_inv_i,
            expr_g_G
        )[None, :, :]
        g_G_left.append(g_G_1)

        # 1 < i < d
        for i in range(2, d):
            expr_U_2 = [0] + [k for k in range(1, d + 1)]
            expr_A_i = [2 * d + 2] + [k for k in range(1, i)] + [2 * d + 3]
            expr_A_inv_i = [2 * d + 4] + [k for k in range(i + 1, d + 1)] + [0]
            expr_g_G = [2 * d + 3, i, 2 * d + 4]

            g_G_i = t.einsum(
                U_2.reshape(r_d + in_shape), expr_U_2,
                TTMatVec.A_i(i-1, *cores), expr_A_i,
                TTMatVec.A_inv_i(d-i, *cores), expr_A_inv_i,
                expr_g_G
            )
            g_G_left.append(g_G_i)

        # i = d
        i = d
        expr_U_2 = [0] + [k for k in range(1, d + 1)]
        expr_A_i = [2 * d + 2] + [k for k in range(1, i)] + [2 * d + 3]
        expr_g_G = [2 * d + 3, i, 0]

        g_G_d = t.einsum(
            U_2.reshape(r_d + in_shape), expr_U_2,
            TTMatVec.A_i(i-1, *cores), expr_A_i,
            expr_g_G
        )
        g_G_left.append(g_G_d)

        # =============

        # g_Gi: i >= d + 1
        T_2 = g_Y[0].T @ T_1
        g_G_right = []

        # i = d + 1
        i = d + 1
        expr_T_2 = [k for k in range(d + 1, 2 * d + 1)] + [d]
        expr_B_inv_i = [2 * d + 3] + [k for k in range(i + 1, 2 * d + 2)]
        expr_g_G = [d, i, 2 * d + 3]

        g_G_d1 = t.einsum(
            T_2.reshape(out_shape + r_d), expr_T_2,
            TTMatVec.B_inv_i(2*d-i, *cores), expr_B_inv_i,
            expr_g_G
        )
        g_G_right.append(g_G_d1)

        # d + 1 < i < 2d
        for i in range(d+2, 2*d):
            expr_T_2 = [k for k in range(d + 1, 2 * d + 1)] + [d]
            expr_B_i = [d] + [k for k in range(d + 1, i)] + [2 * d + 2]
            expr_B_inv_i = [2 * d + 3] + [k for k in range(i + 1, 2 * d + 2)]
            expr_g_G = [2 * d + 2, i, 2 * d + 3]

            g_G_di = t.einsum(
                T_2.reshape(out_shape + r_d), expr_T_2,
                TTMatVec.B_i(i-1-d, *cores), expr_B_i,
                TTMatVec.B_inv_i(2*d-i, *cores), expr_B_inv_i,
                expr_g_G
            )
            g_G_right.append(g_G_di)

        # i = 2d
        i = 2 * d
        expr_T_2 = [k for k in range(d + 1, 2 * d + 1)] + [d]
        expr_B_i = [d] + [k for k in range(d + 1, i)] + [2 * d + 2]
        expr_g_G = [2 * d + 2, i]

        g_G_2d = t.einsum(
            T_2.reshape(out_shape + r_d), expr_T_2,
            TTMatVec.B_i(i-1-d, *cores), expr_B_i,
            expr_g_G
        )[:, :, None]
        g_G_right.append(g_G_2d)

        # =============

        # g = [g_X] + g_G_left + g_G_right
        return g_X, *g_G_left, *g_G_right

In [90]:
class TTMatVecOpt(t.autograd.Function):

    @staticmethod
    def A_i(i: int, *cores: List[t.Tensor]):
        return tt_2_tensor(cores[:i])

    @staticmethod
    def B_i(i: int, *cores: List[t.Tensor]):
        d = len(cores) // 2
        return tt_2_tensor(cores[d:d+i])

    @staticmethod
    def A_inv_i(i: int, *cores: List[t.Tensor]):
        d = len(cores) // 2
        return tt_2_tensor(cores[d-i:d])

    @staticmethod
    def B_inv_i(i: int, *cores: List[t.Tensor]):
        d = len(cores) // 2
        return tt_2_tensor(cores[2*d-i:])

    @staticmethod
    def forward(ctx, X: t.Tensor, *cores: List[t.Tensor]):
        d = len(cores) // 2
        A_d = TTMatVec.A_i(d, *cores)
        B_d = TTMatVec.B_i(d, *cores)
        a_sh = A_d.shape
        b_sh = B_d.shape
        T_1 = X @ A_d.reshape((a_sh[:-1].numel(), a_sh[-1]))
        Y = T_1 @ B_d.reshape((b_sh[0], b_sh[1:].numel()))
        ctx.save_for_backward(X, T_1, *cores)
        return Y

    @staticmethod
    def backward(ctx, *g_Y):
        X, T_1, *cores = ctx.saved_tensors
        d = len(cores) // 2
        r_d = t.Size([cores[d-1].shape[2]])
        in_shape, out_shape = t.Size([]), t.Size([])
        for n in range(d):
            in_shape += t.Size([cores[n].shape[1]])
        for n in range(d, 2*d):
            out_shape += t.Size([cores[n].shape[1]])

        # =============

        # g_X
        A_d = TTMatVec.A_i(d, *cores)
        B_d = TTMatVec.B_i(d, *cores)
        a_sh = A_d.shape
        b_sh = B_d.shape

        U_1 = g_Y[0] @ B_d.reshape((b_sh[0], b_sh[1:].numel())).T
        g_X = U_1 @ A_d.reshape((a_sh[:-1].numel(), a_sh[-1])).T

        # =============

        # g_G_i: i <= d
        U_2 = (U_1.T @ X).reshape(r_d + in_shape)
        g_G_left = []

        # i = 1
        i = 1
        expr_U_2 = [0] + [k for k in range(1, d + 1)]
        expr_A_inv_i = [2 * d + 4] + [k for k in range(i + 1, d + 1)] + [0]
        expr_g_G = [i, 2 * d + 4]

        g_G_1 = t.einsum(
            U_2, expr_U_2,
            TTMatVec.A_inv_i(d-i, *cores), expr_A_inv_i,
            expr_g_G
        )[None, :, :]
        g_G_left.append(g_G_1)

        # 1 < i < d
        for i in range(2, d):
            expr_U_2 = [0] + [k for k in range(1, d + 1)]
            expr_A_i = [2 * d + 2] + [k for k in range(1, i)] + [2 * d + 3]
            expr_A_inv_i = [2 * d + 4] + [k for k in range(i + 1, d + 1)] + [0]
            expr_g_G = [2 * d + 3, i, 2 * d + 4]

            g_G_i = t.einsum(
                U_2, expr_U_2,
                TTMatVec.A_i(i-1, *cores), expr_A_i,
                TTMatVec.A_inv_i(d-i, *cores), expr_A_inv_i,
                expr_g_G
            )
            g_G_left.append(g_G_i)

        # i = d
        i = d
        expr_U_2 = [0] + [k for k in range(1, d + 1)]
        expr_A_i = [2 * d + 2] + [k for k in range(1, i)] + [2 * d + 3]
        expr_g_G = [2 * d + 3, i, 0]

        g_G_d = t.einsum(
            U_2, expr_U_2,
            TTMatVec.A_i(i-1, *cores), expr_A_i,
            expr_g_G
        )
        g_G_left.append(g_G_d)

        # =============

        # g_Gi: i >= d + 1
        T_2 = (g_Y[0].T @ T_1).reshape(out_shape + r_d)
        g_G_right = []

        # i = d + 1
        i = d + 1
        expr_T_2 = [k for k in range(d + 1, 2 * d + 1)] + [d]
        expr_B_inv_i = [2 * d + 3] + [k for k in range(i + 1, 2 * d + 2)]
        expr_g_G = [d, i, 2 * d + 3]

        g_G_d1 = t.einsum(
            T_2, expr_T_2,
            TTMatVec.B_inv_i(2*d-i, *cores), expr_B_inv_i,
            expr_g_G
        )
        g_G_right.append(g_G_d1)

        # d + 1 < i < 2d
        for i in range(d+2, 2*d):
            expr_T_2 = [k for k in range(d + 1, 2 * d + 1)] + [d]
            expr_B_i = [d] + [k for k in range(d + 1, i)] + [2 * d + 2]
            expr_B_inv_i = [2 * d + 3] + [k for k in range(i + 1, 2 * d + 2)]
            expr_g_G = [2 * d + 2, i, 2 * d + 3]

            g_G_di = t.einsum(
                T_2, expr_T_2,
                TTMatVec.B_i(i-1-d, *cores), expr_B_i,
                TTMatVec.B_inv_i(2*d-i, *cores), expr_B_inv_i,
                expr_g_G
            )
            g_G_right.append(g_G_di)

        # i = 2d
        i = 2 * d
        expr_T_2 = [k for k in range(d + 1, 2 * d + 1)] + [d]
        expr_B_i = [d] + [k for k in range(d + 1, i)] + [2 * d + 2]
        expr_g_G = [2 * d + 2, i]

        g_G_2d = t.einsum(
            T_2, expr_T_2,
            TTMatVec.B_i(i-1-d, *cores), expr_B_i,
            expr_g_G
        )[:, :, None]
        g_G_right.append(g_G_2d)

        # =============

        # g = [g_X] + g_G_left + g_G_right
        return g_X, *g_G_left, *g_G_right

In [91]:
class TTLinear(nn.Module):
    def __init__(self, cfg: TTLinearConfig):
        super().__init__()
        self.cfg = cfg
        self.I, self.J = cfg.in_shape.numel(), cfg.out_shape.numel()
        self.cores = nn.ParameterList(self._build_cores())
        self.rank_params = nn.ParameterList(
            self._build_rank()
        ) if cfg.adaptive else None
        self.bias = nn.Parameter(t.zeros(self.J)) if cfg.bias else None

    def _build_cores(self) -> List[nn.Parameter]:
        rank = self.cfg.rank
        shape = self.cfg.in_shape + self.cfg.out_shape
        res = [nn.Parameter(core.to(self.cfg.dtype))
               for core in build_cores_gaus(shape, rank)]
        return res

    def _build_rank(self) -> List[nn.Parameter]:
        rank = self.cfg.rank
        rank_params = [nn.Parameter(t.ones(rank[n]).to(self.cfg.dtype))
                       for n in range(1, self.cfg.N)]
        return rank_params

    def get_rank_mask(self) -> List[t.Tensor]:
        assert self.rank_params is not None, "The mask is only available in adaptive mode"
        mask = []
        threshold = self.cfg.threshold
        for x in self.rank_params:
            y = F.threshold(x, threshold, 0)
            mask.append(y)
        return mask

    def get_cores(self, masked: bool = True) -> List[t.Tensor]:
        if not self.cfg.adaptive or not masked:
            return list(self.cores)
        res = []
        mask = self.get_rank_mask()
        for n in range(len(mask)):
            D = mask[n][None, None, :]
            G = self.cores[n]
            res.append(G * D)
        res.append(self.cores[-1])
        return res

    def forward(self, X):
        out = TTMatVec.apply(X, *self.get_cores())
        return out + self.bias if self.cfg.bias else out

In [48]:
class TTLinearOpt(nn.Module):
    def __init__(self, cfg: TTLinearConfig):
        super().__init__()
        self.cfg = cfg
        self.I, self.J = cfg.in_shape.numel(), cfg.out_shape.numel()
        self.cores = nn.ParameterList(self._build_cores())
        self.rank_params = nn.ParameterList(
            self._build_rank()
        ) if cfg.adaptive else None
        self.bias = nn.Parameter(t.zeros(self.J)) if cfg.bias else None

    def _build_cores(self) -> List[nn.Parameter]:
        rank = self.cfg.rank
        shape = self.cfg.in_shape + self.cfg.out_shape
        res = [nn.Parameter(core.to(self.cfg.dtype))
               for core in build_cores_gaus(shape, rank)]
        return res

    def _build_rank(self) -> List[nn.Parameter]:
        rank = self.cfg.rank
        rank_params = [nn.Parameter(t.ones(rank[n]).to(self.cfg.dtype))
                       for n in range(1, self.cfg.N)]
        return rank_params

    def get_rank_mask(self) -> List[t.Tensor]:
        assert self.rank_params is not None, "The mask is only available in adaptive mode"
        mask = []
        threshold = self.cfg.threshold
        for x in self.rank_params:
            y = F.threshold(x, threshold, 0)
            mask.append(y)
        return mask

    def get_cores(self, masked: bool = True) -> List[t.Tensor]:
        if not self.cfg.adaptive or not masked:
            return list(self.cores)
        res = []
        mask = self.get_rank_mask()
        for n in range(len(mask)):
            D = mask[n][None, None, :]
            G = self.cores[n]
            res.append(G * D)
        res.append(self.cores[-1])
        return res

    def forward(self, X):
        out = TTMatVecOpt.apply(X, *self.get_cores())
        return out + self.bias if self.cfg.bias else out

## Torch compile

In [92]:
rank = t.Size([1, 4, 4, 8, 4, 4, 1])
in_shape = t.Size([4, 8, 8])
out_shape = t.Size([4, 8, 8])
cfg = TTLinearConfig(
    in_shape=in_shape,
    out_shape=out_shape,
    rank=rank,
    adaptive=True,
)
shape = cfg.in_shape + cfg.out_shape

In [93]:
layer = TTLinear(cfg).xpu()
layer_comp_default = t.compile(layer, fullgraph=True)
layer_comp_overhead = t.compile(layer, mode='reduce-overhead')
layer_comp_max = t.compile(layer, mode='max-autotune')

In [94]:
X = t.randn(64, 256).xpu()

In [13]:
# warm up
for i in range(5):
    layer_comp_default(X)
    layer_comp_overhead(X)
    layer_comp_max(X)

c:\dev\adaptive-ttm-gpt\.venv\Lib\site-packages\torch\_inductor\select_algorithm.py:4799: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  current_out_size = out_base.storage().size()
Autotune Choices Stats:
{"num_choices": 14, "num_triton_choices": 13, "best_kernel": "mm", "best_time": 0.022396, "best_triton_pos": 1, "best_triton_time": 0.07953199999999999, "best_triton_kernel": "triton_mm_39", "best_triton_kernel_desc": "ACC_TYPE='tl.float32', ALLOW_TF32=False, BLOCK_K=16, BLOCK_M=64, BLOCK_N=16, EVEN_K=True, GROUP_M=8, USE_FAST_ACCUM=False, num_stages=2, num_warps=4"}
AUTOTUNE mm(64x256, 256x8)
strides: [256, 1], [8, 1]
dtypes: torch.float32, torch.float32
  mm 0.0224 ms 100.0% 
  triton_mm_39 0.0795 ms 28.2% ACC_TYPE='tl.float32', ALLOW_TF32=False,

In [14]:
%%timeit
layer(X)

4.44 ms ± 912 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
%%timeit
layer_comp_default(X)

1.23 ms ± 58.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
%%timeit
layer_comp_overhead(X)

1.2 ms ± 29.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [17]:
%%timeit
layer_comp_max(X)

1.1 ms ± 13.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Naive optimization

In [ ]:
rank = t.Size([1, 4, 4, 8, 4, 4, 1])
in_shape = t.Size([4, 4, 4])
out_shape = t.Size([4, 4, 4])
cfg = TTLinearConfig(
    in_shape=in_shape,
    out_shape=out_shape,
    rank=rank,
    adaptive=False,
    dtype=t.float64
)
shape = cfg.in_shape + cfg.out_shape

In [ ]:
X = t.randn(8, 64).double().xpu()

In [42]:
layer = TTLinear(cfg).xpu()
cores = layer.get_cores(masked=False)

In [43]:
Y = layer(X)
l = t.linalg.norm(Y)
l.backward()

In [44]:
t.autograd.gradcheck(TTMatVec.apply, [X, *cores])

True

In [47]:
t.autograd.gradcheck(TTMatVecOpt.apply, [X, *cores])

True

In [54]:
rank = t.Size([1, 4, 4, 8, 4, 4, 1])
in_shape = t.Size([4, 8, 8])
out_shape = t.Size([4, 8, 8])
cfg = TTLinearConfig(
    in_shape=in_shape,
    out_shape=out_shape,
    rank=rank,
    adaptive=False,
    dtype=t.float32
)
shape = cfg.in_shape + cfg.out_shape
X = t.randn(64, 256).xpu()

In [60]:
layer = TTLinear(cfg).xpu()
layer_opt = TTLinearOpt(cfg).xpu()
layer_comp = t.compile(layer)
layer_opt_comp = t.compile(layer)

In [61]:
# warm up
for i in range(5):
    layer_comp(X)
    layer_opt_comp(X)

In [58]:
%%timeit
Y = layer(X)
l = t.linalg.norm(Y)
l.backward()

8.67 ms ± 160 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [59]:
%%timeit
Y = layer_opt(X)
l = t.linalg.norm(Y)
l.backward()

9.24 ms ± 198 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [63]:
%%timeit
Y = layer_comp(X)
l = t.linalg.norm(Y)
l.backward()

5.01 ms ± 768 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [64]:
%%timeit
Y = layer_opt_comp(X)
l = t.linalg.norm(Y)
l.backward()

4.58 ms ± 126 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Opt Einsum

In [65]:
import opt_einsum as oe

In [66]:
a = t.randn(16, 24)
b = t.randn(24, 8)

In [ ]:
t.einsum(a, [0, 1], b, [1, 2], [0, 2]).shape

torch.Size([16, 8])

In [78]:
path, _ = oe.contract_path(a, [0, 1], b, [1, 2], [0, 2])

In [80]:
oe.contract(a, [0, 1], b, [1, 2], [0, 2], optimize=path).shape

torch.Size([16, 8])